# 🤖 02 - Modeling: Predicting Hospital Readmission

In this notebook, we build and evaluate machine learning models to predict whether a patient will be readmitted within 30 days.

**Goals:**
- Define the target and features
- Train baseline and advanced models
- Evaluate performance with classification metrics
- Compare models and draw conclusions


In [ ]:
# Imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Load cleaned dataset (adjust path as needed)
data_path = '../data/diabetic_data.csv'
df = pd.read_csv(data_path)
df.replace("?", np.nan, inplace=True)


## 🎯 Define Target and Features

In [ ]:
# Create binary target (if not already created)
df['readmitted_binary'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

# Drop columns that shouldn't be features
drop_cols = ['readmitted', 'encounter_id', 'patient_nbr', 'weight', 'payer_code']
df.drop(columns=[col for col in drop_cols if col in df.columns], inplace=True)

# Fill or drop remaining missing values
df.dropna(inplace=True)

# Convert categorical variables
df_encoded = pd.get_dummies(df.drop('readmitted_binary', axis=1), drop_first=True)

X = df_encoded
y = df['readmitted_binary']


## 🧪 Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

## 🔍 Baseline Model: Logistic Regression

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_scaled, y_train)
y_pred_lr = log_reg.predict(X_test_scaled)

print("Logistic Regression Report:")
print(classification_report(y_test, y_pred_lr))
print("AUC:", roc_auc_score(y_test, y_pred_lr))


## 🌲 Advanced Model: XGBoost

In [ ]:
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)

print("XGBoost Classification Report:")
print(classification_report(y_test, y_pred_xgb))
print("AUC:", roc_auc_score(y_test, y_pred_xgb))
